In [175]:
import cv2
import numpy as np

In [176]:
# Implementacion NMS
def nms(boxes, overlap_thresh=0.3):
    if not boxes: return []
    boxes = np.array(boxes)
    pick = []
    x1, y1 = boxes[:,0], boxes[:,1]
    x2, y2 = x1 + boxes[:,2], y1 + boxes[:,3]
    scores = boxes[:,4]
    area = (x2 - x1) * (y2 - y1)
    idxs = np.argsort(scores)

    while len(idxs) > 0:
        last = len(idxs) - 1
        i = idxs[last]
        pick.append(i)
        xx1 = np.maximum(x1[i], x1[idxs[:last]])
        yy1 = np.maximum(y1[i], y1[idxs[:last]])
        xx2 = np.minimum(x2[i], x2[idxs[:last]])
        yy2 = np.minimum(y2[i], y2[idxs[:last]])
        w = np.maximum(0, xx2 - xx1)
        h = np.maximum(0, yy2 - yy1)
        overlap = (w * h) / area[idxs[:last]]
        idxs = np.delete(idxs, np.concatenate(([last], np.where(overlap > overlap_thresh)[0])))
    return boxes[pick]

    

In [177]:
def plot_results(final_detections, img_rgb):
    for (x, y, w, h, score) in final_detections:
        # Dibujar bounding box
        cv2.rectangle(img_rgb, (int(x), int(y)), (int(x + w), int(y + h)), (0, 255, 0), 2)
        # Mostrar nivel de confianza
        text = f"{score:.2f}"
        cv2.putText(img_rgb, text, (int(x), int(y) - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
    cv2.imshow('Detecciones Multiples - Coca Cola', img_rgb)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [178]:
# Mostrar imagenes
def show_image(title, img):
    cv2.imshow(title, img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


In [179]:
# Funcion de deteccion
def detect(th, img_gray, template, scales):
    found_boxes = []

    for scale in scales:
        
        # Calcular nuevas dimensiones
        r_h = int(template.shape[0] * scale) # y
        r_w = int(template.shape[1] * scale) # x
        
        # Si es muy chico, ignorar
        if r_h < 10 or r_w < 10: continue
    
        r_template = cv2.resize(template, (r_w, r_h))
        r_mask =  cv2.resize(mask, (r_w, r_h))
        w, h = r_template.shape[::-1]
    
        #show_image('Template', r_mask)
    
        # print(f"Scaled template : w * h= {w} * {h}")
    
        # Aplicar template matching usando TM_CCOEFF_NORMED al ser robusto a cambios en la iluminacion
        res = cv2.matchTemplate(img_gray, r_mask, cv2.TM_CCOEFF_NORMED)
    
        # Extraer puntos que superen el umbral
        loc = np.where(res >= th)
        
        for pt in zip(*loc[::-1]):
            # Almacenar [x, y, w, h, score]
            score = res[pt[1], pt[0]]
            found_boxes.append([pt[0], pt[1], w, h, score])
    
    fb_bef = len(found_boxes)
        
    found_boxes = nms(found_boxes)
    
    fb_af = len(found_boxes)
    
    print(f"Found boxes before nms : {fb_bef}, after nms: {fb_af}")

    return found_boxes


In [162]:
def process_input(img_rgb):
    img_hsv = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2HSV)
    low_red1 = np.array([0, 70, 50])
    high_red1 = np.array([10, 255, 255])
    low_red2 = np.array([170, 70, 50])
    high_red2 = np.array([180, 255, 255])
    mask_red1 = cv2.inRange(img_hsv, low_red1, high_red1)
    mask_red2 = cv2.inRange(img_hsv, low_red2, high_red2)
    mask_red = cv2.add(mask_red1, mask_red2)
    return cv2.bitwise_and(img_gray, img_gray, mask=mask_red)

In [180]:
# Definir constantes
IMAGE_FOLDER='./images'
IMAGE = 'coca_multi.png'
TEMPLATE_FOLDER= './template'
TEMPLATE='pattern.png'

In [181]:
# Leer archivos

# Archivo original (color)
img_file = IMAGE_FOLDER + '/' + IMAGE
img_rgb = cv2.imread(img_file)
# Escala de grises
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2GRAY)
# Cargar el template y procesarlo de tal manera de dejar las letras blancas y el fondo negro para que al momento de intentar matchear
# el fondo no incremente las posibilidades de obtener falsos positivo
template = cv2.imread(TEMPLATE_FOLDER + '/' + TEMPLATE, 0)
_, template = cv2.threshold(template, 200, 255, cv2.THRESH_BINARY_INV)

print(f"template dimensions: w * h = {template.shape[1]} * {template.shape[0]}")
print(f"image dimensions: w * h = {img_gray.shape[1]} * {img_gray.shape[0]}")

template dimensions: w * h = 400 * 175
image dimensions: w * h = 799 * 598


In [182]:
# Se observa que el archivo con el patron a buscar es mas grande que los logos que se ven en la imagen a procesar. 
# De ahi que se plantea un re escalamiento
scales = np.linspace(0.1, 0.45, 30) 

# Umbral de confinza a utilizar (de las pruebas dio que este es el valor adecuado)
th = 0.48

In [185]:
# Ejecutar la deteccion
found_boxes = detect(th, img_gray, template, scales)

Found boxes before nms : 26, after nms: 10


In [186]:
# Plotear los resultados sobre la imagen original
plot_results(found_boxes, img_rgb)

In [ ]:
# Nota: Lo que se observo fue que si el th se baja, aparecen matches erroneos sobre los estantes que donde estan apoyadas las botellas
# Por falta de tiempo no lo hago pero tal vez se podria intentar tratar estas partes de la imagen para removerlas y asi lograr poder bajar
# el th y posiblemente lograr mas detecciones de los otros logos que no se estan detectando producto de la deformacion que tienen